**1. Import Required Libraries**

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

**2. Load the 5G NIDD Dataset**

In [ ]:
# Load
df = pd.read_csv('/content/drive/MyDrive/5G_NIDD_multiclass_clean.csv', low_memory=False)

print("Original shape:", df.shape)

# Target
y = df['Label']
X = df.drop(columns=['Label', 'Attack Type', 'Attack Tool'], errors='ignore')

# Remove obvious non-learning columns
drop_cols = [
    'SrcMac','DstMac','SrcAddr','DstAddr','StartTime','LastTime',        # drop these columns as they dont have any importance in model(these are IP addresses)
    'SrcOui','DstOui'
]

X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')

# Keep only numeric features
X = X.select_dtypes(include=[np.number])   # keeps only numeric features

print("After numeric selection:", X.shape)

Original shape: (1215890, 112)
After numeric selection: (1215890, 86)


**3. Data Cleaning and Handling Missing Values**

In [ ]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)     # remove infinite values
X.fillna(0, inplace=True)                              # handle missing values

**4. Feature Selection using SelectKBest**

In [ ]:
selector = SelectKBest(score_func=f_classif, k=36)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected Features:", selected_features.tolist())

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 1  7 13 14 19 20 21 22 23 24 25 26 27 34 35 36 37 48 49 50 51 57 58 59
 60 61 62 63 64 65 66 67 68 69 70 71 72 77 78 79 80] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Selected Features: ['Rank', 'Seq', 'Dur', 'RunTime', 'Mean', 'Sum', 'Min', 'Max', 'sTos', 'dTos', 'sTtl', 'dTtl', 'sHops', 'dHops', 'TotPkts', 'SrcPkts', 'DstPkts', 'TotBytes', 'SrcBytes', 'DstBytes', 'Offset', 'sMeanPktSz', 'dMeanPktSz', 'Loss', 'SrcLoss', 'DstLoss', 'pLoss', 'SrcWin', 'DstWin', 'sVid', 'dVid', 'SrcTCPBase', 'DstTCPBase', 'TcpRtt', 'SynAck', 'AckDat']


**5. Label Encoding(Encode Target Labels)**

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_classes = len(np.unique(y_encoded))
print("Classes:", num_classes)

Classes: 20


**6. Split Dataset into Training and Testing Sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.2,                    # train data = 80%, test data = 20%
    stratify=y_encoded,
    random_state=42
)

**7. Feature Scaling using QuantileTransformer**

In [ ]:
from sklearn.preprocessing import QuantileTransformer

scaler = QuantileTransformer(
    n_quantiles=min(1000, len(X_train)),   # avoids warning if dataset < 1000
    output_distribution='normal',          # 'normal' usually works better for neural networks
    random_state=42
)

X_train = scaler.fit_transform(X_train)   # FIT ONLY TRAIN
X_test  = scaler.transform(X_test)        # TRANSFORM TEST

**8. Reshape Data for BiGRU Input**

In [ ]:
X_train = X_train.reshape(-1, 36, 1)        # Prepare data in 3D format required by GRU networks
X_test  = X_test.reshape(-1, 36, 1)

**9. Import Deep Learning Layers and Mobile-Net-V1 & BIGRU(Combined) MODEL**

In [ ]:
def MobileNetV1_BiGRU(drop_rate, gru_units, dense_units):

    inp = Input(shape=(36,1))

    # ----- CNN BRANCH -----                    # MOBILE-NET V1 MODEL
    x = Reshape((36,1,1))(inp)

    x = Conv2D(32,(3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(64,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(128,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    cnn_out = GlobalAveragePooling2D()(x)

    # ----- GRU BRANCH -----                                 # BIGRU MODEL
    r = Bidirectional(GRU(gru_units, return_sequences=True))(inp)
    r = Bidirectional(GRU(gru_units))(r)

    # ----- FUSION -----                                    # PROJECTION(Combines both models)
    merged = Concatenate()([cnn_out, r])

    merged = Dense(dense_units, activation="relu")(merged)
    merged = Dense(dense_units//2, activation="relu")(merged)
    merged = Dropout(drop_rate)(merged)

    out = Dense(num_classes, activation="softmax")(merged)

    model = Model(inp, out)
    return model


**10. Install keras tuner**

In [ ]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 12.3 MB/s eta 0:00:00


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.3 MB/s eta 0:00:00


**11. Setup TensorFlow & Focal Loss**

In [ ]:
import tensorflow as tf

def focal_loss(gamma, alpha):

    def loss(y_true, y_pred):

        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.one_hot(y_true, depth=num_classes)

        ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)

        pt = tf.exp(-ce)
        focal = alpha * tf.pow(1 - pt, gamma) * ce

        return tf.reduce_mean(focal)

    return loss


**12. Handle Class Imbalance using Class Weights**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights))
print(class_weights)


{np.int64(0): np.float64(0.64811172410117), np.int64(1): np.float64(0.6491671115856914), np.int64(2): np.float64(3.325965944060726), np.int64(3): np.float64(4.20650406504065), np.int64(4): np.float64(37.819284603421465), np.int64(5): np.float64(44.74296228150874), np.int64(6): np.float64(1.3619983757596124), np.int64(7): np.float64(4.309374446216552), np.int64(8): np.float64(5.309563318777292), np.int64(9): np.float64(5.274438781043271), np.int64(10): np.float64(1.9601644365629534), np.int64(11): np.float64(4.803516049382716), np.int64(12): np.float64(5.220652640618291), np.int64(13): np.float64(5.217292426517915), np.int64(14): np.float64(1.5948189926547744), np.int64(15): np.float64(1.9197000197355436), np.int64(16): np.float64(0.12998123867505493), np.int64(17): np.float64(0.21242149215139894), np.int64(18): np.float64(6.053721682847897), np.int64(19): np.float64(5.8995147986414365)}


100% DATA = 80% TRAIN | 20% TEST  (test=used for final evaluation)

TRAIN DATA = used for hypertuning and divided into clients

**13. Optuna Training of hyperparameters**

In [ ]:
import optuna
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

X_tune = X_train
y_tune = y_train


def objective(trial):

    tf.keras.backend.clear_session()

    dropout = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
    gru_units = trial.suggest_int("gru_units", 128, 256, step=64)
    dense_units = trial.suggest_int("dense_units", 256, 384, step=64)
    gamma = trial.suggest_float("gamma", 1.0, 3.0, step=1)
    alpha = trial.suggest_float("alpha", 0.2, 0.4, step=0.1)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 10, 20, step=10)

    model = MobileNetV1_BiGRU(
        drop_rate=dropout,
        gru_units=gru_units,
        dense_units=dense_units
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=gamma, alpha=alpha),
        metrics=["accuracy"]
    )

    history = model.fit(
        X_tune,
        y_tune,
        validation_split=0.2,
        epochs=epochs,
        batch_size=256,
        callbacks=[
            EarlyStopping(patience=3, restore_best_weights=True),
            ReduceLROnPlateau(patience=2)
        ],
        verbose=1
    )

    return max(history.history["val_accuracy"])


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

best_params = study.best_params

[I 2026-04-14 04:26:59,524] A new study created in memory with name: no-name-af75a99a-7dac-470c-ad55-47bc3ca42351


Epoch 1/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 114s 32ms/step - accuracy: 0.8188 - loss: 0.0580 - val_accuracy: 0.8691 - val_loss: 0.0360 - learning_rate: 0.0035
Epoch 2/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 94s 31ms/step - accuracy: 0.8606 - loss: 0.0391 - val_accuracy: 0.8875 - val_loss: 0.0327 - learning_rate: 0.0035
Epoch 3/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 94s 31ms/step - accuracy: 0.8701 - loss: 0.0356 - val_accuracy: 0.8689 - val_loss: 0.0390 - learning_rate: 0.0035
Epoch 4/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 94s 31ms/step - accuracy: 0.8752 - loss: 0.0346 - val_accuracy: 0.8847 - val_loss: 0.0293 - learning_rate: 0.0035
Epoch 5/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 142s 31ms/step - accuracy: 0.8769 - loss: 0.0338 - val_accuracy: 0.8804 - val_loss: 0.0310 - learning_rate: 0.0035
Epoch 6/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 94s 31ms/step - accuracy: 0.8688 - loss: 0.0372 - val_accuracy: 0.8522 - val_loss: 0.0430 - learning_rate: 0.0035
Epoch 7/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 94s 31ms/step - accu

[I 2026-04-14 05:01:46,485] Trial 0 finished with value: 0.9459297060966492 and parameters: {'dropout': 0.4, 'gru_units': 128, 'dense_units': 320, 'gamma': 2.0, 'alpha': 0.30000000000000004, 'lr': 0.003504190426464412, 'epochs': 20}. Best is trial 0 with value: 0.9459297060966492.


Epoch 1/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 141s 44ms/step - accuracy: 0.7807 - loss: 0.1450 - val_accuracy: 0.8845 - val_loss: 0.0721 - learning_rate: 3.2338e-04
Epoch 2/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 132s 43ms/step - accuracy: 0.8791 - loss: 0.0688 - val_accuracy: 0.8995 - val_loss: 0.0565 - learning_rate: 3.2338e-04
Epoch 3/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.8929 - loss: 0.0581 - val_accuracy: 0.8989 - val_loss: 0.0502 - learning_rate: 3.2338e-04
Epoch 4/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.9012 - loss: 0.0525 - val_accuracy: 0.9045 - val_loss: 0.0501 - learning_rate: 3.2338e-04
Epoch 5/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.9075 - loss: 0.0490 - val_accuracy: 0.9195 - val_loss: 0.0433 - learning_rate: 3.2338e-04
Epoch 6/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.9117 - loss: 0.0462 - val_accuracy: 0.9138 - val_loss: 0.0422 - learning_rate: 3.2338e-04
Epoch 7/20
3040/3040 ━━━━━━━━━━━━━

[I 2026-04-14 05:46:31,369] Trial 1 finished with value: 0.9808782339096069 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 192, 'dense_units': 320, 'gamma': 1.0, 'alpha': 0.4, 'lr': 0.00032337881948552255, 'epochs': 20}. Best is trial 1 with value: 0.9808782339096069.


...

[I 2026-04-14 06:05:09,581] Trial 2 finished with value: 0.9261911511421204 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 128, 'dense_units': 256, 'gamma': 2.0, 'alpha': 0.2, 'lr': 0.0009060908015607059, 'epochs': 10}. Best is trial 1 with value: 0.9808782339096069.


Epoch 1/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 140s 43ms/step - accuracy: 0.8193 - loss: 0.0610 - val_accuracy: 0.8682 - val_loss: 0.0337 - learning_rate: 7.5590e-04
Epoch 2/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 131s 43ms/step - accuracy: 0.8847 - loss: 0.0306 - val_accuracy: 0.8981 - val_loss: 0.0266 - learning_rate: 7.5590e-04
Epoch 3/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 132s 43ms/step - accuracy: 0.8944 - loss: 0.0266 - val_accuracy: 0.9051 - val_loss: 0.0243 - learning_rate: 7.5590e-04
Epoch 4/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.9035 - loss: 0.0239 - val_accuracy: 0.9123 - val_loss: 0.0216 - learning_rate: 7.5590e-04
Epoch 5/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.9079 - loss: 0.0224 - val_accuracy: 0.9002 - val_loss: 0.0255 - learning_rate: 7.5590e-04
Epoch 6/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 132s 44ms/step - accuracy: 0.9137 - loss: 0.0210 - val_accuracy: 0.9143 - val_loss: 0.0182 - learning_rate: 7.5590e-04
Epoch 7/10
3040/3040 ━━━━━━━━━━━━━

[I 2026-04-14 06:27:23,272] Trial 3 finished with value: 0.9326112866401672 and parameters: {'dropout': 0.2, 'gru_units': 192, 'dense_units': 384, 'gamma': 2.0, 'alpha': 0.30000000000000004, 'lr': 0.0007559014679154209, 'epochs': 10}. Best is trial 1 with value: 0.9808782339096069.


Epoch 1/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 109s 33ms/step - accuracy: 0.8185 - loss: 0.1139 - val_accuracy: 0.8507 - val_loss: 0.0934 - learning_rate: 0.0077
Epoch 2/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 142s 33ms/step - accuracy: 0.8371 - loss: 0.0986 - val_accuracy: 0.8284 - val_loss: 0.1070 - learning_rate: 0.0077
Epoch 3/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 100s 33ms/step - accuracy: 0.8410 - loss: 0.0955 - val_accuracy: 0.8510 - val_loss: 0.0918 - learning_rate: 0.0077
Epoch 4/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 141s 33ms/step - accuracy: 0.8445 - loss: 0.0936 - val_accuracy: 0.8304 - val_loss: 0.1458 - learning_rate: 0.0077
Epoch 5/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 144s 33ms/step - accuracy: 0.8491 - loss: 0.0904 - val_accuracy: 0.8658 - val_loss: 0.0753 - learning_rate: 0.0077
Epoch 6/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 99s 33ms/step - accuracy: 0.8459 - loss: 0.0931 - val_accuracy: 0.8638 - val_loss: 0.0785 - learning_rate: 0.0077
Epoch 7/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - a

[I 2026-04-14 07:02:44,775] Trial 4 finished with value: 0.9484432935714722 and parameters: {'dropout': 0.4, 'gru_units': 128, 'dense_units': 256, 'gamma': 1.0, 'alpha': 0.4, 'lr': 0.007661811995737931, 'epochs': 20}. Best is trial 1 with value: 0.9808782339096069.


**14. Compiled model**

In [ ]:
def get_compiled_model():
    model = MobileNetV1_BiGRU(
        drop_rate=0.2,
        gru_units=192,
        dense_units=320
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0009661918592170282),
        loss=focal_loss(gamma=1.0, alpha=0.4),
        metrics=["accuracy"]
    )

    return model

**15. LWE Key Generation**

In [ ]:
import numpy as np

# Default (will be tuned)
LWE_PARAMS = {
    "n": 64,
    "q": 1021,
    "sigma": 2.0,
    "scale": 1000
}

def generate_keys(params):
    n, q, sigma = params["n"], params["q"], params["sigma"]

    A = np.random.randint(0, q, (n, n))
    s = np.random.randint(0, q, (n, 1))
    e = np.random.normal(0, sigma, (n, 1)).astype(int)

    b = (A @ s + e) % q
    return (A, b), s


def encrypt(pub, m, params):
    A, b = pub
    n, q = params["n"], params["q"]

    r = np.random.randint(0, 2, (n, 1))

    c1 = (A.T @ r) % q
    c2 = (b.T @ r + (q//2) * m) % q

    return (c1, c2)

def decrypt(sec, ct, params):
    c1, c2 = ct
    q = params["q"]

    val = (c2 - (c1.T @ sec)) % q

    # Recover integer message properly
    return int(np.round(val.item() / (q//2)))

**16. Encrypt Neural Network Weights**

In [ ]:
def quantize_weights(weights, scale):
    return [(w * scale).astype(int) for w in weights]

def dequantize_weights(weights, scale):
    return [w / scale for w in weights]

**17. Create Federated Clients**

In [ ]:
from sklearn.model_selection import StratifiedKFold

NUM_CLIENTS = 3

def create_clients(X, y, num_clients):
    skf = StratifiedKFold(n_splits=num_clients, shuffle=True, random_state=42)
    client_data = {}

    for i, (_, idx) in enumerate(skf.split(X, y)):
        client_data[i] = (X[idx], y[idx])

    return client_data


client_data = create_clients(X_train, y_train, NUM_CLIENTS)

**18. Client Local Training Function**

In [ ]:
def client_update(global_weights, dataset, local_epochs=3):

    local_model = get_compiled_model()
    local_model.set_weights(global_weights)

    X, y = dataset

    local_model.fit(
        X,
        y,
        epochs=local_epochs,
        batch_size=256,
        verbose=0
    )

    return local_model.get_weights(), len(X)

**19. Federated Averaging (FedAvg)**

In [ ]:
def aggregate_weights(client_weights, client_sizes):

    total_samples = sum(client_sizes)
    avg_weights = []

    for weights_list_tuple in zip(*client_weights):
        weighted_sum = np.zeros_like(weights_list_tuple[0])

        for w, size in zip(weights_list_tuple, client_sizes):
            weighted_sum += w * (size / total_samples)

        avg_weights.append(weighted_sum)

    return avg_weights

**18. Federated Training Loop (Server)**

In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 3

global_model = get_compiled_model()

# RANDOM initialization (IMPORTANT)
global_weights = global_model.get_weights()

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    global_weights = aggregate_weights(client_weights, client_sizes)
    global_model.set_weights(global_weights)


FL Round 0

FL Round 1

FL Round 2

FL Round 3

FL Round 4

FL Round 5

FL Round 6

FL Round 7

FL Round 8

FL Round 9

FL Round 10

FL Round 11

FL Round 12


**19. Final Accuracy**

In [ ]:
loss, accuracy = global_model.evaluate(X_test, y_test)
print("Final Global Model Accuracy:", accuracy)

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 59s 8ms/step - accuracy: 0.9726 - loss: 0.0190
Final Global Model Accuracy: 0.9726455807685852


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# 🔹 Get predictions
y_pred_probs = global_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 If y_test is one-hot encoded
if len(y_test.shape) > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# 🔹 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# 🔹 Classification Report (precision, recall, f1 per class)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# 🔹 Overall Metrics
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 43s 6ms/step

Confusion Matrix:
 [[18517    51     8     9     6     6    50     3     0     0    15    30
      1     2    21     5    29     6     1     1]
 [    7 18444     5    20     4    31    24     4     5     0    68    67
      0     0     9     2    12    25     2     1]
 [   12    15  3398    66     7     1    47     9     1     0     2     8
      0     2    41     0    38     1     5     3]
 [    0    11    26  2690    29    30     5     1     1     1     5    11
      0     5     4     1    38    21     0    11]
 [    0     0    15    12   214    41     1     7     0     0     0     5
      0     0     0     0    21     2     0     4]
 [    0     3     1     6    33   220     0     1     0     0     0     4
      0     0     0     1     1     1     0     0]
 [    3     6    30     7     9     7  8735    10     1     2    24    33
      0     2     0    15    28    13     1     1]
 [    1     0     0     1     0     4    18  2750     0     0

In [ ]:
# For binary classification only
if len(np.unique(y_true)) == 2:

    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn)   # Recall / Sensitivity
    fpr = fp / (fp + tn)

    print(f"\nTPR (Recall): {tpr:.4f}")
    print(f"FPR: {fpr:.4f}")

In [ ]:
num_classes = cm.shape[0]

for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0

    print(f"\nClass {i}: TPR={tpr:.4f}, FPR={fpr:.4f}")


Class 0: TPR=0.9870, FPR=0.0004

Class 1: TPR=0.9847, FPR=0.0010

Class 2: TPR=0.9294, FPR=0.0011

Class 3: TPR=0.9308, FPR=0.0010

Class 4: TPR=0.6646, FPR=0.0006

Class 5: TPR=0.8118, FPR=0.0011

Class 6: TPR=0.9785, FPR=0.0015

Class 7: TPR=0.9745, FPR=0.0003

Class 8: TPR=0.9332, FPR=0.0008

Class 9: TPR=0.9063, FPR=0.0006

Class 10: TPR=0.9541, FPR=0.0013

Class 11: TPR=0.8882, FPR=0.0022

Class 12: TPR=0.7544, FPR=0.0005

Class 13: TPR=0.8756, FPR=0.0018

Class 14: TPR=0.9641, FPR=0.0004

Class 15: TPR=0.9915, FPR=0.0003

Class 16: TPR=0.9746, FPR=0.0028

Class 17: TPR=0.9919, FPR=0.0123

Class 18: TPR=0.9507, FPR=0.0013

Class 19: TPR=0.9151, FPR=0.0004


In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 5

global_model = get_compiled_model()

# RANDOM initialization (IMPORTANT)
global_weights = global_model.get_weights()

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    global_weights = aggregate_weights(client_weights, client_sizes)
    global_model.set_weights(global_weights)


FL Round 0

FL Round 1

FL Round 2

FL Round 3

FL Round 4

FL Round 5

FL Round 6

FL Round 7

FL Round 8

FL Round 9

FL Round 10

FL Round 11

FL Round 12


In [ ]:
loss, accuracy = global_model.evaluate(X_test, y_test)
print("Final Global Model Accuracy:", accuracy)

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 72s 9ms/step - accuracy: 0.9444 - loss: 0.0369
Final Global Model Accuracy: 0.9443617463111877


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# 🔹 Get predictions
y_pred_probs = global_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 If y_test is one-hot encoded
if len(y_test.shape) > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# 🔹 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# 🔹 Classification Report (precision, recall, f1 per class)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# 🔹 Overall Metrics
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 52s 7ms/step

Confusion Matrix:
 [[18448    68    16    10     6     5    64     0     0     0    16    21
      2     7    43     9    35    10     1     0]
 [    3 18391     9    18     7    24    23     4     2     2    97    60
      0     6    12     1    10    59     1     1]
 [    1    16  3374    75     4     1    44     9     0     0     5    10
      0     0    84     0    24     5     4     0]
 [    2    16    33  2677    30    13     2     1     0     1     2    15
      0     0    15     1    43    35     0     4]
 [    2     0    17    14   200    48     3     8     0     0     0     1
      0     0     0     0    22     7     0     0]
 [    0     0     1    22    24   203     1     2     0     0     0     2
      0     0     0     1     1    14     0     0]
 [    0     6    81    16     5     4  8684    10     0     6    37     8
      0    11     8    12    24    10     2     3]
 [    0     1     1     1     0     2    12  2743     0     1

In [ ]:
# For binary classification only
if len(np.unique(y_true)) == 2:

    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn)   # Recall / Sensitivity
    fpr = fp / (fp + tn)

    print(f"\nTPR (Recall): {tpr:.4f}")
    print(f"FPR: {fpr:.4f}")

In [ ]:
num_classes = cm.shape[0]

for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0

    print(f"\nClass {i}: TPR={tpr:.4f}, FPR={fpr:.4f}")


Class 0: TPR=0.9833, FPR=0.0002

Class 1: TPR=0.9819, FPR=0.0012

Class 2: TPR=0.9229, FPR=0.0015

Class 3: TPR=0.9263, FPR=0.0015

Class 4: TPR=0.6211, FPR=0.0005

Class 5: TPR=0.7491, FPR=0.0007

Class 6: TPR=0.9728, FPR=0.0016

Class 7: TPR=0.9720, FPR=0.0003

Class 8: TPR=0.9170, FPR=0.0004

Class 9: TPR=0.9258, FPR=0.0009

Class 10: TPR=0.9420, FPR=0.0027

Class 11: TPR=0.7499, FPR=0.0019

Class 12: TPR=0.7553, FPR=0.0018

Class 13: TPR=0.7602, FPR=0.0022

Class 14: TPR=0.9637, FPR=0.0008

Class 15: TPR=0.9912, FPR=0.0004

Class 16: TPR=0.9404, FPR=0.0209

Class 17: TPR=0.9459, FPR=0.0304

Class 18: TPR=0.8716, FPR=0.0008

Class 19: TPR=0.9393, FPR=0.0007


In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 7

global_model = get_compiled_model()

# RANDOM initialization (IMPORTANT)
global_weights = global_model.get_weights()

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    global_weights = aggregate_weights(client_weights, client_sizes)
    global_model.set_weights(global_weights)


FL Round 0

FL Round 1

FL Round 2

FL Round 3

FL Round 4

FL Round 5

FL Round 6

FL Round 7

FL Round 8

FL Round 9

FL Round 10

FL Round 11

FL Round 12


In [ ]:
loss, accuracy = global_model.evaluate(X_test, y_test)
print("Final Global Model Accuracy:", accuracy)

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 60s 8ms/step - accuracy: 0.9379 - loss: 0.0322
Final Global Model Accuracy: 0.9378890991210938


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# 🔹 Get predictions
y_pred_probs = global_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 If y_test is one-hot encoded
if len(y_test.shape) > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# 🔹 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# 🔹 Classification Report (precision, recall, f1 per class)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# 🔹 Overall Metrics
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 42s 6ms/step

Confusion Matrix:
 [[18470    71    16     6     8     5    39     1     0     1    12    33
      2     7    25     5    37    10     8     5]
 [    1 18344    17    56     8    22    33     4     2     1    84    57
      0     3     3     1    25    67     0     2]
 [   15     8  3419    80     7     1    38     8     0     0     2    12
      0     0    27     0    31     3     3     2]
 [    0    14    32  2669    19     9     1     4     0     2     1     5
      0     5     6     2    63    31     0    27]
 [    2     0    16    18   187    55     0     7     0     0     0     2
      0     0     0     0    21     7     0     7]
 [    1     0     1     6    25   196     0     1     0     0     3    15
      0     0     0     2    15     6     0     0]
 [    2     6    66    11    10     2  8685    10     0     4    17    39
      0    10     2    12    33    13     2     3]
 [    1     1     2     2     1     1   328  2435     0     0

In [ ]:
# For binary classification only
if len(np.unique(y_true)) == 2:

    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn)   # Recall / Sensitivity
    fpr = fp / (fp + tn)

    print(f"\nTPR (Recall): {tpr:.4f}")
    print(f"FPR: {fpr:.4f}")

In [ ]:
num_classes = cm.shape[0]

for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0

    print(f"\nClass {i}: TPR={tpr:.4f}, FPR={fpr:.4f}")


Class 0: TPR=0.9845, FPR=0.0005

Class 1: TPR=0.9794, FPR=0.0012

Class 2: TPR=0.9352, FPR=0.0016

Class 3: TPR=0.9235, FPR=0.0016

Class 4: TPR=0.5807, FPR=0.0005

Class 5: TPR=0.7232, FPR=0.0007

Class 6: TPR=0.9729, FPR=0.0028

Class 7: TPR=0.8629, FPR=0.0004

Class 8: TPR=0.8441, FPR=0.0007

Class 9: TPR=0.8924, FPR=0.0011

Class 10: TPR=0.9420, FPR=0.0026

Class 11: TPR=0.7408, FPR=0.0019

Class 12: TPR=0.6539, FPR=0.0016

Class 13: TPR=0.7902, FPR=0.0034

Class 14: TPR=0.9567, FPR=0.0004

Class 15: TPR=0.9847, FPR=0.0003

Class 16: TPR=0.9100, FPR=0.0076

Class 17: TPR=0.9814, FPR=0.0455

Class 18: TPR=0.9174, FPR=0.0011

Class 19: TPR=0.9345, FPR=0.0007


In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 9

global_model = get_compiled_model()

# RANDOM initialization (IMPORTANT)
global_weights = global_model.get_weights()

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    global_weights = aggregate_weights(client_weights, client_sizes)
    global_model.set_weights(global_weights)


FL Round 0

FL Round 1

FL Round 2

FL Round 3

FL Round 4

FL Round 5

FL Round 6

FL Round 7

FL Round 8

FL Round 9

FL Round 10

FL Round 11

FL Round 12


In [ ]:
loss, accuracy = global_model.evaluate(X_test, y_test)
print("Final Global Model Accuracy:", accuracy)

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 61s 8ms/step - accuracy: 0.9288 - loss: 0.0378
Final Global Model Accuracy: 0.9288175702095032


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# 🔹 Get predictions
y_pred_probs = global_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 If y_test is one-hot encoded
if len(y_test.shape) > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# 🔹 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# 🔹 Classification Report (precision, recall, f1 per class)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# 🔹 Overall Metrics
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 60s 8ms/step

Confusion Matrix:
 [[18417    75    15    10    12     4    59     1     0     0    13    22
      4     1    39    30    55     3     1     0]
 [    2 18374    10    57    14    28    45     6     2     0    52    80
      0     0     3     1    26    30     0     0]
 [   13    13  3413    82     7     1    14    10     0     1     7    13
      0     0    35     1    44     2     0     0]
 [    2    12    56  2633    35    11     1     3     0     4     0    14
      0     0     8     1    75    26     0     9]
 [    2     1    15    13   208    44     0     7     0     0     0     0
      0     0     0     0    26     6     0     0]
 [    0     0     2     7    44   186     0     3     0     0     2     1
      0     0     0     2    18     6     0     0]
 [    8    12    38    14     9     3  8672    10     0     4    26     8
      0     5    56    14    31    13     2     2]
 [    0     1     4     1     0     5    19  2747     0     1

In [ ]:
# For binary classification only
if len(np.unique(y_true)) == 2:

    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn)   # Recall / Sensitivity
    fpr = fp / (fp + tn)

    print(f"\nTPR (Recall): {tpr:.4f}")
    print(f"FPR: {fpr:.4f}")

In [ ]:
num_classes = cm.shape[0]

for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0

    print(f"\nClass {i}: TPR={tpr:.4f}, FPR={fpr:.4f}")


Class 0: TPR=0.9817, FPR=0.0003

Class 1: TPR=0.9810, FPR=0.0014

Class 2: TPR=0.9335, FPR=0.0015

Class 3: TPR=0.9111, FPR=0.0018

Class 4: TPR=0.6460, FPR=0.0008

Class 5: TPR=0.6863, FPR=0.0007

Class 6: TPR=0.9714, FPR=0.0018

Class 7: TPR=0.9734, FPR=0.0004

Class 8: TPR=0.7515, FPR=0.0042

Class 9: TPR=0.5453, FPR=0.0019

Class 10: TPR=0.9337, FPR=0.0028

Class 11: TPR=0.7365, FPR=0.0020

Class 12: TPR=0.7020, FPR=0.0015

Class 13: TPR=0.7816, FPR=0.0020

Class 14: TPR=0.9559, FPR=0.0008

Class 15: TPR=0.9889, FPR=0.0005

Class 16: TPR=0.9053, FPR=0.0149

Class 17: TPR=0.9635, FPR=0.0476

Class 18: TPR=0.9104, FPR=0.0010

Class 19: TPR=0.9282, FPR=0.0005
